# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution – Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library. All dataset elements—record sets, fields, and columns—are referenced by their `@id` per best practice.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading

First, we'll load the dataset metadata and instantiate a `Dataset` object from the Croissant schema URL. This will allow further exploration using the mlcroissant API.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Dataset description: {metadata.description}\n")
print(f"Cite as: {metadata.cite_as}")

## 2. Data Overview

Let's review the available record sets and their corresponding fields using their `@id`. This provides a programmatic overview of the dataset structure for subsequent extraction and analysis.

**Note:** Importantly, we always reference record sets, fields, and columns by their `@id` values.

In [ ]:
# List all record sets defined in the Croissant metadata
record_sets = list(dataset.record_sets)
print("Record sets (@id):")
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs.get('name','(no name)')}")

# For the main data, let's assume the largest (most fields) record set holds the primary dataset
# Print fields for each record set
from collections import defaultdict
fields_per_rs = defaultdict(list)

for rs in record_sets:
    fields = dataset.fields(record_set=rs['@id'])
    print(f"\nFields in record set @id `{rs['@id']}`:")
    for field in fields:
        print(f"  - @id: {field['@id']} | name: {field.get('name','(no name)')} | dataType: {field.get('dataType', '(none)')}")
        fields_per_rs[rs['@id']].append(field)

## 3. Data Extraction

Load data from each record set into separate pandas DataFrames. Use the `@id` of each record set as the reference.

*Below we extract all record sets, storing them by their `@id` in a dictionary for easy reference.*

In [ ]:
dataframes = {}

for rs in record_sets:
    rs_id = rs['@id']
    print(f"Extracting records for record set @id: {rs_id}")
    # Each record is a dict keyed by field @id
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"  Loaded {len(records)} records. Columns (field @ids):\n    {', '.join(dataframes[rs_id].columns)}")
    else:
        print("  No records extracted.")

# Pick main record set (assume largest resulting DataFrame)
main_rs_id = max(dataframes, key=lambda k: len(dataframes[k].columns))
print(f"\nMain tabular data record set id: {main_rs_id}\n")
print("Preview of the data columns (@id):")
print(dataframes[main_rs_id].columns.tolist())
display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Now let's process and explore the main tabular data. We'll demonstrate:
- Filtering records by a numeric field using its `@id`.
- Normalizing a numeric field.
- Grouping data by a key attribute for summary.

### Finding Numeric and Categorical Fields
We'll select a numeric field and a grouping (categorical) field by examining the fields in the main record set.

In [ ]:
# List all fields and try to identify a numeric and grouping field by @id
main_rs_fields = fields_per_rs[main_rs_id]
numeric_field_id = None
group_field_id = None

# Try to find numeric field (@id) and grouping field (@id)
for f in main_rs_fields:
    dt = str(f.get('dataType','')).lower()
    if ('float' in dt or 'integer' in dt or 'number' in dt) and not numeric_field_id:
        numeric_field_id = f['@id']
    if ('sex' in f.get('name','').lower() or 'gender' in f.get('name','').lower() or 'anatomical' in f.get('name','').lower()) and not group_field_id:
        group_field_id = f['@id']

print(f"Inferred numeric field for EDA: {numeric_field_id}")
print(f"Inferred group (categorical) field: {group_field_id}")

df = dataframes[main_rs_id]
# If columns are stringified numbers, coerce
if numeric_field_id in df.columns:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Show basic info
print("DataFrame info and preview:")
display(df.info())
display(df[[numeric_field_id, group_field_id]].head())

In [ ]:
# Filtering records based on a numeric field
if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id]).all() else 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where `{numeric_field_id}` > {threshold:.2f}: {len(filtered_df)} rows\n")
    display(filtered_df[[numeric_field_id, group_field_id]].head())

    # Normalize the selected field
    filtered_df[f'{numeric_field_id}_normalized'] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized `{numeric_field_id}` for filtered records:")
    display(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

    # Grouped statistics by the group field, if present
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nAverage `{numeric_field_id}` grouped by `{group_field_id}`:")
        display(grouped_df)

## 5. Visualization

Let's visualize distributions or relationships between key fields using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
if numeric_field_id is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot grouped by the grouping field if available
    if group_field_id in df.columns and len(df[group_field_id].dropna().unique()) > 1:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- This notebook demonstrated how to load and explore a Croissant-described clinical dataset programmatically using `mlcroissant`.
- All data entities ([record sets](https://mlcommons.github.io/croissant/record-set.html), fields, columns) were referenced by their `@id` ensuring clarity and reproducibility.
- Analytics steps included filtering, normalization, grouping, and visualizing with pandas/Seaborn.

With this workflow, you can further analyze the FAIR2 dataset or adapt these steps for other Croissant-compatible datasets.